# Spectral Joint Flow (SJFPO) — Online Colab / Kaggle (with internet)

Clone the repo with a GitHub token, install requirements, then run training
via `!python scripts/train.py`. No source is embedded in the notebook.

## 0. Secrets and environment

1. Set `GITHUB_TOKEN` in the environment (Colab secrets or Kaggle secrets).
2. The token is read via `load_secret()` using the runtime environment.
3. Run all cells top to bottom.

In [ ]:
# Detect runtime and load secrets from the right store.
import os

OUTPUT_DIR = {
    "colab": "/content/",
    "kaggle": "/kaggle/working/",
    "local": "./",
}
INPUT_DIR = {
    "colab": "/content/",
    "kaggle": "/kaggle/input/",
    "local": "./",
}
def detect_env():
    try:
        if "google.colab" in str(get_ipython()):
            return "colab"
    except NameError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    return "local"

ENV_NAME = detect_env()
print(f"Environment: {ENV_NAME}")

def load_secret(key_name):
    try:
        if ENV_NAME == "colab":
            from google.colab import userdata
            return userdata.get(key_name) or None
        if ENV_NAME == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key_name) or None
        return os.getenv(key_name)
    except Exception as e:
        print(f"load_secret({key_name}) failed: {e}")
        return None

ENV_NAME = detect_env()


In [ ]:
# Clone the repo using the GitHub token from the environment/store.
%cd $OUTPUT_DIR[ENV_NAME]
!rm -rf spectral-joint-policy
GITHUB_TOKEN = load_secret("GITHUB_TOKEN")
repo_slug = os.getenv("GITHUB_REPO", "dzungphieuluuky/spectral-joint-policy")
repo_url = f"https://github.com/{repo_slug}.git"
if GITHUB_TOKEN:
    repo_url = repo_url.replace("github.com", f"{GITHUB_TOKEN}@github.com")
print(f"Cloning from: {repo_url.replace(GITHUB_TOKEN or '', '<TOKEN>') if GITHUB_TOKEN else repo_url}")
!git clone $repo_url spectral-joint-policy

In [ ]:
# Move into the cloned repo and run directly (no embedded source).
%cd spectral-joint-policy
!pwd

In [ ]:
%%writefile requirements-colab.txt
# Deep Learning
torch
torchvision
torchaudio
torchcodec
jax
jaxlib
accelerate
diffusers
datasets
huggingface-hub
safetensors
bitsandbytes
einops
einx

# Reinforcement Learning / Robot Learning
gymnasium
gymnasium-robotics
pettingzoo
mujoco
skrl
lerobot

# Scientific / Vision
numpy
scipy
pandas
matplotlib
scikit-learn
scikit-image
opencv-python-headless
pillow
imageio
imageio-ffmpeg
av
moviepy

# Data / Storage
pyarrow

# Experiment Tracking / Config
wandb
tensorboard
draccus


In [ ]:
# Detect runtime and choose install source.
if ENV_NAME == "colab":
    !uv pip install --upgrade -r requirements-colab.txt
else:
    !uv pip install --upgrade -r requirements-colab.txt

In [ ]:
# wandb login from secret store (runs before any training cell).
_wandb_key = load_secret("WANDB_API_KEY")
if _wandb_key:
    !wandb login $_wandb_key
else:
    print("WANDB_API_KEY not found; training will run without wandb sync.")

In [ ]:
# Run training directly from the cloned repo.
!SJFPO_COMPILE=0 python scripts/train.py     --method sjfpo     --env Hopper-v5     --seed 3407     --total_steps 200000     --eval_steps 50     --save_steps 10000     --sjfpo.use_per_bin_clip true     --sjfpo.top_p_spectral_tokens 1.0

In [ ]:
# Run training directly from the cloned repo.
!SJFPO_COMPILE=0 python scripts/train.py     --method sjfpo     --env Hopper-v5     --seed 3407     --total_steps 200000     --eval_steps 50     --save_steps 10000     --sjfpo.use_per_bin_clip true     --sjfpo.top_p_spectral_tokens 1.0